# Yield Curve Visualization

Visualize the U.S. municipal bond yield curve over time using MSRB trade data.
Aggregates weekly average yields by `time_to_maturity` (in years) and renders as a heatmap, contour plot, and 3D surface.

**Source**: `nyu-datasets.munibonds.trades_typed`

## Setup

Authenticate via Application Default Credentials, or service account key:
```bash
export GOOGLE_APPLICATION_CREDENTIALS=/path/to/key.json
```

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from scipy.interpolate import griddata
import seaborn as sns

%config InlineBackend.figure_format = 'retina'

client = bigquery.Client(project='nyu-datasets')

## Query: weekly avg yield by time-to-maturity

Buckets trades into integer years-to-maturity (0..30), then averages yield within each (week, maturity) cell.
Filters out cells with fewer than 100 trades to reduce noise.

In [ ]:
QUERY = """
SELECT
  DATE_TRUNC(TRADE_DATE, WEEK) AS trade_date,
  DATE_DIFF(MATURITY_DATE, TRADE_DATE, YEAR) AS time_to_maturity,
  AVG(YIELD) AS avg_yield,
  COUNT(YIELD) AS cnt_yield
FROM `nyu-datasets.munibonds.trades_typed`
WHERE TRADE_DATE < MATURITY_DATE
  AND YIELD IS NOT NULL
  AND TRADE_DATE IS NOT NULL
  AND MATURITY_DATE IS NOT NULL
GROUP BY trade_date, time_to_maturity
HAVING time_to_maturity BETWEEN 0 AND 30 AND cnt_yield > 100
ORDER BY trade_date, time_to_maturity
"""

df = client.query(QUERY).to_dataframe()
df['trade_date'] = pd.to_datetime(df['trade_date'])
df['avg_yield'] = pd.to_numeric(df['avg_yield'])
df['time_to_maturity'] = df['time_to_maturity'].astype(int)
df = df.query('time_to_maturity > 0')
print(f'{len(df):,} (week, maturity) cells from {df.trade_date.min():%Y-%m-%d} to {df.trade_date.max():%Y-%m-%d}')
df.head()

## Heatmap

In [ ]:
heatmap = df.pivot_table(
    index='time_to_maturity',
    columns='trade_date',
    values='avg_yield',
    aggfunc='mean',
)

plt.figure(figsize=(24, 8))
sns.heatmap(heatmap.iloc[::-1], cbar_kws={'label': 'Yield (%)'}, cmap='YlGnBu')
plt.title('Weekly Avg Yield by Time-to-Maturity')
plt.tight_layout()
plt.show()

## Smoothed contour plot

Interpolates the (week, maturity, yield) grid for a smoother visualization.

In [ ]:
df['day_offset'] = (df['trade_date'] - df['trade_date'].min()).dt.days

x = np.linspace(df.day_offset.min(), df.day_offset.max(), len(df.day_offset.unique()))
y = np.linspace(df.time_to_maturity.min(), df.time_to_maturity.max(), len(df.time_to_maturity.unique()))
X, Y = np.meshgrid(x, y)
Z = griddata(
    points=(df.day_offset, df.time_to_maturity),
    values=df.avg_yield,
    xi=(X, Y),
    method='nearest',
)

fig, ax = plt.subplots(figsize=(20, 12))
ax.contourf(X, Y, Z, 200, cmap=cm.YlGnBu)
ax.set_title(f'Yield Curve as a Function of Maturity Over Time ({df.trade_date.min():%b %Y} - {df.trade_date.max():%b %Y})')
ax.set_xlabel('Date')
ax.set_ylabel('Time to Maturity (years)')

first_year = df.trade_date.min().year
last_year = df.trade_date.max().year
ax.xaxis.set_major_locator(ticker.MultipleLocator(365))
ax.set_xticklabels(range(first_year, last_year + 1))

ax.grid(True, color='k', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

## 3D surface

In [ ]:
fig, ax = plt.subplots(figsize=(20, 12), subplot_kw={'projection': '3d'})
ax.set_title(f'Yield Curve Surface ({df.trade_date.min():%b %Y} - {df.trade_date.max():%b %Y})')
ax.set_xlabel(f'Days since {df.trade_date.min():%Y-%m-%d}')
ax.set_ylabel('Time to Maturity (years)')
ax.set_zlabel('Yield (%)')

ax.plot_surface(X, Y, Z, cmap=cm.YlGnBu, rstride=1, cstride=1, alpha=0.9, edgecolors='k', lw=0.05)
ax.set_zlim(0, max(8, np.nanmax(Z)))
ax.zaxis.set_major_locator(LinearLocator(13))
ax.zaxis.set_major_formatter(FormatStrFormatter('%.02f'))
plt.tight_layout()
plt.show()